# Máscara de Navio com Detecção via CFAR

## CFAR

In [10]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
from pathlib import Path
from scipy.ndimage import uniform_filter, median_filter, label, binary_dilation, binary_fill_holes

def read_tif(path):
    with rasterio.open(path) as src:
        img = src.read(1).astype(np.float32)
        profile = src.profile.copy()
    return img, profile

def save_tif(path, img, profile):
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(img, 1)

def db_to_linear(img_db):
    return 10 ** (img_db / 10.0)

def linear_to_db(img_linear):
    return 10 * np.log10(np.maximum(img_linear, 1e-10))

# CFAR CA (Cell-Averaging CFAR)

def cfar_ca(img, train=10, guard=3, pfa=1e-4):
    """
    CA-CFAR correto usando convolução de soma acumulada.
    Funciona em domínio linear (intensidade, não dB).
    """
    power = img.copy()
    power = np.where(power <= 0, 1e-10, power)

    outer_size = 2 * (train + guard) + 1
    inner_size = 2 * guard + 1

    # Soma acumulada da janela externa e interna
    sum_outer = uniform_filter(power, size=outer_size) * (outer_size ** 2)
    sum_inner = uniform_filter(power, size=inner_size) * (inner_size ** 2)

    n_outer = outer_size ** 2
    n_inner = inner_size ** 2
    N = n_outer - n_inner  # número de células de treino

    # Média do clutter apenas nas células de treino (excluindo guard)
    clutter = (sum_outer - sum_inner) / N

    # Fator de threshold para distribuição exponencial (intensidade SAR)
    alpha = N * (pfa ** (-1.0 / N) - 1)

    threshold = alpha * clutter

    mask = power > threshold

    print(f"[CFAR] N={N}, alpha={alpha:.4f}")
    print(f"[CFAR] clutter: min={clutter.min():.4f}, max={clutter.max():.4f}, mean={clutter.mean():.4f}")
    print(f"[CFAR] threshold: min={threshold.min():.4f}, max={threshold.max():.4f}")
    print(f"[CFAR] detecções brutas: {mask.sum()} pixels ({100*mask.sum()/mask.size:.3f}%)")

    return mask, threshold

# FILTRAGEM POR ÁREA

def filter_objects(mask, min_area=5, max_area=5000):
    labels, nobj = label(mask)
    output = np.zeros_like(mask)
    for i in range(1, nobj + 1):
        area = np.sum(labels == i)
        if min_area <= area <= max_area:
            output[labels == i] = True
    print(f"[FILTRO] objetos antes: {nobj}, depois do filtro de área: {output.any() and label(output)[1]}")
    return output

def expand_mask(mask, iterations=5):
    return binary_dilation(mask, iterations=iterations)

def fill_mask(mask):
    return binary_fill_holes(mask)

def remove_targets_median(img, mask, window=11):
    filtered = median_filter(img, size=window)
    output = img.copy()
    output[mask] = filtered[mask]
    return output

def mask_statistics(mask):
    labels, nobj = label(mask)
    areas = [np.sum(labels == i) for i in range(1, nobj + 1)]
    return {"n_objects": nobj, "areas": areas, "total_pixels": int(np.sum(mask))}

# PIPELINE COMPLETO

def ship_mask_pipeline(img, train=10, guard=3, pfa=1e-4,
                       min_area=5, max_area=5000, dilation=5):

    mask, threshold = cfar_ca(img, train=train, guard=guard, pfa=pfa)
    mask = filter_objects(mask, min_area=min_area, max_area=max_area)
    mask = fill_mask(mask)
    mask = expand_mask(mask, iterations=dilation)
    return mask

## Salvar PNG

In [11]:
def _rotate_and_crop(arr: np.ndarray, heading_deg: Optional[float],
                     footprint: Optional[np.ndarray] = None) -> np.ndarray:
    """Retorna `arr` recortado ao bounding-box da pegada SAR (sem rotação).

    A rotação por -heading foi removida: cria cantos NaN triangulares que
    aparentam "cortes" no campo de vento em relação à cena bruta. Preserva
    a orientação N-up original do raster SNAP e apenas recorta bordas
    puramente NaN usando `footprint` (ou `isfinite(arr)` como fallback).
    """
    a = np.array(arr, dtype=np.float64)
    fp = footprint
    if fp is not None:
        fp = np.asarray(fp, dtype=bool)
        if fp.shape != a.shape:
            fp = None
    crop_mask = fp if fp is not None else np.isfinite(a)
    if crop_mask.any():
        rows = np.any(crop_mask, axis=1)
        cols = np.any(crop_mask, axis=0)
        y0, y1 = np.argmax(rows), len(rows) - np.argmax(rows[::-1])
        x0, x1 = np.argmax(cols), len(cols) - np.argmax(cols[::-1])
        a = a[y0:y1, x0:x1]
    return a


def save_png(path: Path, data: np.ndarray,
             heading_deg: Optional[float] = None,
             label: Optional[str] = None,
             unit: Optional[str] = None,
             stats: Optional[dict] = None,
             footprint: Optional[np.ndarray] = None,
             vmin: Optional[float] = None,
             vmax: Optional[float] = None) -> None:
    """Salva PNG com colorbar, crop pelo footprint e escala de cor.

    footprint: máscara 2D (shape igual a `data`) indicando a pegada da cena
    SAR. Define o bounding box do crop.

    vmin/vmax: limites de cor explícitos. Quando passados, NÃO recalcula
    (essencial para comparar PNGs da mesma cena com pré-processamentos
    diferentes, ex. várias janelas de suavização).
    """
    arr = _rotate_and_crop(data, heading_deg, footprint=footprint)
    mask = np.isfinite(arr)
    if not np.any(mask):
        plt.imsave(str(path), np.zeros_like(arr, dtype=np.float32))
        return

    if vmin is None or vmax is None or not (vmax > vmin):
        vmin = float(np.nanpercentile(arr[mask], 2))
        vmax = float(np.nanpercentile(arr[mask], 98))
        if vmax <= vmin:
            vmin = float(np.nanmin(arr[mask]))
            vmax = float(np.nanmax(arr[mask]))
            if vmax <= vmin:
                vmax = vmin + 1.0

    h, w = arr.shape
    fig_w = 8.0
    fig_h = max(3.0, min(12.0, 8.0 * h / max(w, 1)))
    fig, ax = plt.subplots(figsize=(fig_w + 1.5, fig_h), dpi=150)
    im = ax.imshow(arr, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()

    title = label or Path(path).stem
    if unit:
        title = f"{title} [{unit}]"
    ax.set_title(title, fontsize=11)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cbar.ax.tick_params(labelsize=9)
    if unit:
        cbar.set_label(unit, fontsize=10)

    if stats:
        txt = (
            f"n={stats.get('n', 0):,}  "
            f"min={stats.get('min', float('nan')):.2f}  "
            f"max={stats.get('max', float('nan')):.2f}\n"
            f"mean={stats.get('mean', float('nan')):.2f}  "
            f"median={stats.get('median', float('nan')):.2f}  "
            f"std={stats.get('std', float('nan')):.2f}"
        )
        ax.text(0.01, -0.02, txt, transform=ax.transAxes, fontsize=8,
                va="top", ha="left", family="monospace",
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="gray", pad=3))

    fig.tight_layout()
    fig.savefig(str(path), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)


## Main

In [12]:
img, profile = read_tif(r"SC_20231108T052911\12_ICEYE_X11_GRD_SC_2937021_20231108T052911_sigma.tif")

img_proc = img.copy() # já está em linear normalizado
nonzero_min = img_proc[img_proc > 0].min()
img_proc[img_proc == 0] = nonzero_min

out_dir = Path(r"SC_20231108T052911")

mask = ship_mask_pipeline(
    img_proc,
    train=24,
    guard=4,
    pfa=1e-2,
    min_area=1,
    max_area=2000,
    dilation=8
)

# ── Salvar máscara binária ──────────────────────────

mask_profile = {**profile, "dtype": "uint8", "count": 1}

save_tif(
    out_dir / "mask_navios_SC_20231108T052911.tif",
    mask.astype(np.uint8),
    mask_profile
)

print("Máscara salva: mask_navios_SC_20231108T052911.tif")

# ── Remover navios com mediana e salvar ─────────────

img_clean = remove_targets_median(
    img,
    mask,
    window=25
)

save_tif(
    out_dir / "sigma0_sem_navios_SC_20231108T052911.tif",
    img_clean.astype(np.float32),
    profile
)

print("Imagem limpa salva: sigma0_sem_navios_SC_20231108T052911.tif")

save_png(out_dir / "sigma0_com_navios_SC_20231108T052911.png", img.astype(np.float32),
         label="Imagem com Navios",
         unit="binário", stats=mask_statistics(mask))
save_png(out_dir / "sigma0_sem_navios_SC_20231108T052911.png", img_clean.astype(np.float32),
         label="Imagem limpa (CFAR CA)",
         unit="linear", stats=mask_statistics(mask))

print(mask_statistics(mask))



[CFAR] N=3168, alpha=4.6085
[CFAR] clutter: min=0.0069, max=0.7642, mean=0.1334
[CFAR] threshold: min=0.0318, max=3.5219
[CFAR] detecções brutas: 164 pixels (0.008%)
[FILTRO] objetos antes: 105, depois do filtro de área: 105
Máscara salva: mask_navios_SC_20231108T052911.tif
Imagem limpa salva: sigma0_sem_navios_SC_20231108T052911.tif
{'n_objects': 63, 'areas': [334, 145, 563, 145, 319, 290, 145, 145, 145, 311, 162, 342, 145, 145, 218, 145, 145, 162, 145, 145, 179, 179, 180, 180, 220, 205, 145, 145, 179, 162, 145, 145, 145, 145, 179, 223, 171, 196, 145, 145, 371, 145, 162, 196, 171, 180, 145, 368, 162, 162, 217, 162, 178, 188, 145, 145, 145, 145, 145, 145, 239, 145, 190], 'total_pixels': 11960}
